# Passo 4: Processamento de Alta Performance e Gravação de Vídeo

Você percebeu o maior problema da Visão Computacional para web/notebooks: tentar desenhar imagem por imagem na tela do navegador (`matplotlib`) é o que causa a lentidão absurda.

Como seu computador tem poder de processamento, a forma **correta e profissional** de se trabalhar com vídeos longos é:
1. Ler os frames na velocidade máxima que a CPU conseguir.
2. Passar a Inteligência Artificial (MediaPipe).
3. Gravar o resultado (frame já com as linhas desenhadas) direto no disco rígido criando um arquivo de vídeo (`.mp4`).

Sem a trava do navegador atualizando a tela, o processamento voa!

In [2]:
import cv2
import mediapipe as mp
import time

# IMPORTANTE: Cole sua URL novamente
RTSP_URL = 'rtsp://admin:@Jeff2712@10.64.0.61:554/'

# --- SETUP DA IA ---
mp_desenho = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

hands = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Conectando na câmera
cap = cv2.VideoCapture(RTSP_URL)

# Pegamos as dimensões nativas do vídeo da sua câmera
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = 30 # Vamos salvar o arquivo a 30 FPS

# --- SETUP DO GRAVADOR DE VÍDEO ---
# Usaremos o codec mp4v que gera arquivos .mp4 compatíveis com players tradicionais
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
nome_arquivo = 'resultado_ia.mp4'
out = cv2.VideoWriter(nome_arquivo, fourcc, fps, (largura, altura))

# Vamos gravar 300 frames (~10 segundos contínuos)
NUM_FRAMES = 300

print(f"Iniciando gravação de {NUM_FRAMES} frames. Pode fazer os movimentos na frente da câmera!")
tempo_inicio = time.time()

try:
    for i in range(NUM_FRAMES):
        ret, frame = cap.read()
        if not ret:
            print("Falha ao ler da câmera no frame", i)
            break
            
        # Converte para a IA analisar
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        resultado_maos = hands.process(frame_rgb)
        resultado_corpo = pose.process(frame_rgb)
        
        # Se achar mãos, desenha no frame original (BGR, pois o gravador de vídeo usa BGR)
        if resultado_maos.multi_hand_landmarks:
            for mao in resultado_maos.multi_hand_landmarks:
                mp_desenho.draw_landmarks(frame, mao, mp_hands.HAND_CONNECTIONS)
                
        # Se achar corpo, desenha também
        if resultado_corpo.pose_landmarks:
            mp_desenho.draw_landmarks(frame, resultado_corpo.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # Salva a imagem desenhada dentro do arquivo de vídeo no disco rígido
        out.write(frame)
        
        # Avisa o progresso a cada 50 frames para você saber que não travou
        if i % 50 == 0:
            print(f"-> Processados {i}/{NUM_FRAMES} frames...")
            
finally:
    # É obrigatório liberar os arquivos, senão o vídeo final fica corrompido!
    cap.release()
    out.release()
    hands.close()
    pose.close()
    tempo_fim = time.time()
    print(f"\nPronto! Vídeo salvo com sucesso como '{nome_arquivo}'.")
    print(f"O processamento completo levou {tempo_fim - tempo_inicio:.1f} segundos.")

Iniciando gravação de 300 frames. Pode fazer os movimentos na frente da câmera!
-> Processados 0/300 frames...
-> Processados 50/300 frames...
-> Processados 100/300 frames...


[hevc @ 0x608286f6b440] Could not find ref with POC 22


-> Processados 150/300 frames...


[hevc @ 0x608286f8b940] Could not find ref with POC 19
[hevc @ 0x6082870c7480] Could not find ref with POC 6


-> Processados 200/300 frames...


[hevc @ 0x60828b396880] Could not find ref with POC 41


-> Processados 250/300 frames...


[hevc @ 0x608286f6b440] Could not find ref with POC 24
[hevc @ 0x608286f7b6c0] Could not find ref with POC 56



Pronto! Vídeo salvo com sucesso como 'resultado_ia.mp4'.
O processamento completo levou 21.0 segundos.


### Como assistir?
Olhe na barra lateral esquerda aqui do Jupyter (onde ficam os arquivos). 
Você verá um arquivo novo chamado **`resultado_ia.mp4`**. 

Basta dar **duplo clique** para abrir ou clicar com o **botão direito -> Download** para baixar para o seu PC e assistir no seu reprodutor de vídeo normal, em velocidade máxima e sem os engasgos do navegador!